# Codificação de Variáveis Categóricas - Parte 1
## Conceitos e Técnicas Fundamentais

**Disciplina:** INF1032 - Introdução à Ciência de Dados  
**Professor:** Adriano Branco  
**Instituição:** PUC-Rio - Departamento de Informática  

### Aula 08 - Parte 1

---

### 🎯 **Objetivos desta parte:**
- Compreender o desafio de trabalhar com variáveis categóricas em ML
- Aprender as técnicas fundamentais: Label, One-Hot e Binary Encoding
- Implementar cada técnica usando Python e scikit-learn
- Comparar vantagens e desvantagens de cada método

---

## 1. Introdução à Codificação de Variáveis Categóricas

### POR QUE VARIÁVEIS CATEGÓRICAS SÃO UM DESAFIO?


**1. Tipos de variáveis categóricas:**
   - **Nominal**: Categorias sem ordem intrínseca (ex: cores, cidades, gêneros musicais)
   - **Ordinal**: Categorias com ordem natural (ex: pequeno/médio/grande, níveis educacionais)
   - **Binária**: Apenas duas categorias (ex: sim/não, verdadeiro/falso)
   
**2. Algoritmos de ML trabalham com números, não categorias**
   - A maioria dos algoritmos de ML opera com entradas numéricas
   - Algoritmos como regressão linear, redes neurais e SVM não processam texto diretamente
   - Precisamos converter categorias para representações numéricas


**3. Desafios específicos:**
   - Preservar relações entre categorias (quando relevantes)
   - Evitar falsas relações ordinais
   - Lidar com alta cardinalidade (muitas categorias diferentes)
   - Gerenciar categorias raras/novas em dados de teste
   - Evitar explosão de dimensionalidade (muitas colunas geradas)

**4. Impacto em diferentes algoritmos:**
   - Alguns algoritmos (ex: árvores de decisão) podem funcionar com categorias codificadas de forma simples
   - Outros (ex: regressão linear, SVM) requerem codificações mais elaboradas
   - Codificação inadequada pode degradar significativamente o desempenho

## 2. Criando o Dataset de Exemplo

Vamos criar um conjunto de dados sintético que simula informações de um e-commerce, contendo variáveis categóricas de diferentes tipos e cardinalidades.

In [ ]:
# @title
# Importações necessárias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('default')
sns.set_palette('viridis')

# Fixando a semente para reprodutibilidade
np.random.seed(42)
n = 1000

# Criando um DataFrame com variáveis categóricas e numéricas
df = pd.DataFrame({
    # Variáveis categóricas
    'sexo': np.random.choice(['Masculino', 'Feminino'], n, p=[0.45, 0.55]),
    'estado_civil': np.random.choice(['Solteiro', 'Casado', 'Divorciado', 'Viúvo'], n, p=[0.35, 0.45, 0.15, 0.05]),
    'nivel_educacao': np.random.choice(['Ensino Fundamental', 'Ensino Médio', 'Graduação',
                                       'Pós-graduação', 'Mestrado', 'Doutorado'], n,
                                      p=[0.15, 0.30, 0.30, 0.15, 0.07, 0.03]),
    'categoria_produto': np.random.choice(['Eletrônicos', 'Moda', 'Casa e Decoração', 'Saúde e Beleza',
                                          'Esportes', 'Livros', 'Alimentos', 'Brinquedos',
                                          'Ferramentas', 'Automotivo'], n),
    'cidade': np.random.choice(['São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Salvador', 'Brasília',
                               'Recife', 'Fortaleza', 'Curitiba', 'Manaus', 'Porto Alegre', 'Campinas',
                               'Goiânia', 'Belém', 'Florianópolis', 'Vitória'], n),
    'meio_pagamento': np.random.choice(['Cartão de Crédito', 'Boleto', 'Pix',
                                       'Transferência Bancária', 'Cartão de Débito'], n,
                                      p=[0.40, 0.15, 0.30, 0.05, 0.10]),
    'faixa_renda': np.random.choice(['Até R$2.000', 'R$2.001 a R$4.000',
                                    'R$4.001 a R$8.000', 'Acima de R$8.000'], n,
                                   p=[0.25, 0.35, 0.25, 0.15]),
    'canal_aquisicao': np.random.choice(['Busca Orgânica', 'Redes Sociais', 'Email Marketing',
                                        'Indicação', 'Anúncios'], n,
                                       p=[0.25, 0.30, 0.15, 0.15, 0.15]),

    # Variáveis numéricas
    'idade': np.random.beta(2, 2.5, n) * 60 + 18,  # Idade entre 18 e 78 anos
    'valor_compra': np.random.lognormal(5.5, 0.9, n),  # Valores de compra com distribuição lognormal
    'tempo_site_minutos': np.random.gamma(2, 5, n),  # Tempo no site
    'compras_anteriores': np.random.negative_binomial(3, 0.5, n),  # Número de compras anteriores
    'itens_carrinho': np.random.poisson(3, n) + 1,  # Itens no carrinho (mínimo 1)
})

# Criando variável alvo binária baseada em múltiplos fatores
probabilidade_compra = (
    0.3 * (df['compras_anteriores'] > 2).astype(int) +
    0.2 * (df['valor_compra'] > df['valor_compra'].median()).astype(int) +
    0.2 * (df['tempo_site_minutos'] > 10).astype(int) +
    0.15 * (df['meio_pagamento'] == 'Cartão de Crédito').astype(int) +
    0.15 * np.random.random(n)
)
df['nova_compra'] = (probabilidade_compra > 0.5).astype(int)

# Arredondando valores numéricos
df['valor_compra'] = df['valor_compra'].round(2)
df['tempo_site_minutos'] = df['tempo_site_minutos'].round(2)
df['idade'] = df['idade'].round(0)

print(f"Dataset gerado com {n} registros e {len(df.columns)} variáveis")
print(f"\nPrimeiras 5 linhas:")
df.head()

In [ ]:
# @title
# Analisando a distribuição das variáveis categóricas
print("="*60)
print("ANÁLISE DAS VARIÁVEIS CATEGÓRICAS")
print("="*60)

for col in df.select_dtypes(include=['object']).columns[:3]:  # Mostrando apenas 3 para economia de espaço
    print(f"\nDistribuição de {col}:")
    counts = df[col].value_counts()
    percentages = df[col].value_counts(normalize=True) * 100
    dist_df = pd.DataFrame({'Contagem': counts, 'Percentual (%)': percentages})
    print(dist_df)

# Analisando cardinalidade
print("\n" + "="*60)
print("CARDINALIDADE DAS VARIÁVEIS CATEGÓRICAS")
print("="*60)
for col in df.select_dtypes(include=['object']).columns:
    print(f"{col}: {df[col].nunique()} categorias únicas")

## 3. Label Encoding

O **Label Encoding** é a técnica mais simples para transformar variáveis categóricas em numéricas. Cada categoria única recebe um número inteiro.

### Como funciona:
- Atribui um número inteiro único para cada categoria
- A atribuição geralmente é feita em ordem alfabética
- Ex: ["gato", "cachorro", "pássaro"] → [1, 0, 2]

### Quando usar:
- Variáveis ordinais com ordem natural
- Algoritmos baseados em árvores (Random Forest, XGBoost)
- Quando a simplicidade é importante

### Cuidados:
- Cria relação ordinal artificial em variáveis nominais
- Pode confundir algoritmos baseados em distância

In [ ]:
# @title
# Implementando Label Encoding
from sklearn.preprocessing import LabelEncoder

# Criando uma cópia do DataFrame
df_label = df.copy()

# Aplicando Label Encoding em várias colunas
for col in ['sexo', 'nivel_educacao', 'categoria_produto']:
    le = LabelEncoder()
    df_label[f'{col}_encoded'] = le.fit_transform(df_label[col])

    # Mostrando o mapeamento para cada coluna
    print(f"\nMapeamento para '{col}':")
    unique_values = df_label[col].unique()
    encoded_values = le.transform(unique_values)
    for orig, enc in zip(unique_values, encoded_values):
        print(f"  {orig} → {enc}")

In [ ]:
# @title
# Visualizando o resultado do Label Encoding
sample_cols = ['sexo', 'sexo_encoded', 'nivel_educacao', 'nivel_educacao_encoded',
              'categoria_produto', 'categoria_produto_encoded']
print("\nExemplo de Label Encoding aplicado:")
df_label[sample_cols].head(10)

### Análise do Label Encoding

Observe que o Label Encoding:
- ✅ **Mantém apenas uma coluna** por variável categórica
- ⚠️ **Cria uma ordem artificial** entre as categorias
- ⚠️ **Pode induzir o modelo a interpretar** que Masculino (0) < Feminino (1)
- 📊 **Ideal para variáveis ordinais** onde a ordem importa

## 4. One-Hot Encoding

O **One-Hot Encoding** cria uma coluna binária para cada categoria única, evitando a criação de relações ordinais artificiais.

### Como funciona:
- Cria uma nova coluna binária para cada categoria
- Apenas uma coluna terá valor 1 para cada registro
- Usamos `drop='first'` para evitar multicolinearidade

### Vantagens:
- Elimina a relação ordinal artificial do Label Encoding
- Representa corretamente variáveis nominais
- Cada categoria se torna uma feature independente

### Desvantagens:
- Aumenta significativamente o número de colunas
- Pode causar problemas com alta cardinalidade

In [ ]:
# @title
# Implementando One-Hot Encoding
from sklearn.preprocessing import OneHotEncoder

# Criando uma cópia do DataFrame
df_onehot = df.copy()

# Aplicando One-Hot Encoding
cat_cols_to_encode = ['sexo', 'estado_civil', 'categoria_produto','nivel_educacao']

# Usando OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop='first')
onehot_encoded = ohe.fit_transform(df_onehot[cat_cols_to_encode])

# Criando nomes para as novas colunas
feature_names = []
for i, col in enumerate(cat_cols_to_encode):
    categories = ohe.categories_[i][1:]  # drop='first' remove a primeira
    for cat in categories:
        feature_names.append(f"{col}_{cat}")

# Convertendo para DataFrame
df_onehot_encoded = pd.DataFrame(onehot_encoded, columns=feature_names, index=df_onehot.index)

# Concatenando com o DataFrame original
df_onehot_final = pd.concat([df_onehot, df_onehot_encoded], axis=1)

print(f"One-Hot Encoding aplicado:")
print(f"Colunas originais: {len(cat_cols_to_encode)}")
print(f"Colunas após One-Hot: {len(feature_names)}")
print(f"\nPrimeiras linhas do resultado:")
#df_onehot_final[['sexo'] + [col for col in feature_names if 'sexo' in col]].head()
df_onehot_final[['nivel_educacao'] + [col for col in feature_names if 'nivel_educacao' in col]].head()


### Análise do One-Hot Encoding

O One-Hot Encoding criou colunas binárias para cada categoria:
- ✅ **Eliminou a falsa ordem** do Label Encoding  
- ⚠️ **Aumentou dimensionalidade**: 4 colunas viraram 18 colunas
- 💡 Usamos `drop='first'` para evitar multicolinearidade
- 📊 **Trade-off**: Melhor representação vs. mais dimensões

## 5. Binary Encoding

O **Binary Encoding** é uma técnica que combina elementos do Label Encoding e One-Hot Encoding, convertendo categorias em representação binária.

### Como funciona:
1. Atribui um número a cada categoria (como no Label Encoding)
2. Converte cada número para representação binária
3. Cria uma coluna para cada bit da representação

### Exemplo:
Para categorias ["A", "B", "C", "D"]:
- Label Encoding: [0, 1, 2, 3]
- Representação binária: ["00", "01", "10", "11"]
- Resulta em 2 colunas binárias

### Vantagens:
- Requer menos colunas que One-Hot Encoding (logarítmico vs. linear)
- Melhor para variáveis com alta cardinalidade
- Equilibra entre compactação e representação

In [ ]:
# @title
# Instalando category_encoders se necessário
try:
    from category_encoders import BinaryEncoder
except ImportError:
    !pip install category_encoders -q
    from category_encoders import BinaryEncoder

# Criando uma cópia do DataFrame
df_binary = df.copy()

# Selecionando uma coluna de alta cardinalidade para demonstração
col_high_card = 'cidade'

# Aplicando Binary Encoding
binary_encoder = BinaryEncoder(cols=[col_high_card])
binary_encoded = binary_encoder.fit_transform(df_binary[col_high_card])

# Visualizando o resultado
print(f"\nCodificação Binária para '{col_high_card}':")
print(f"Número de categorias únicas: {df_binary[col_high_card].nunique()}")
print(f"Número de colunas binárias criadas: {binary_encoded.shape[1]}")
print("\nPrimeiras linhas com codificação binária:")
comparison_binary = pd.concat([df_binary[col_high_card].reset_index(drop=True), binary_encoded], axis=1)
comparison_binary.head(10)

### Análise do Binary Encoding

O Binary Encoding oferece um meio termo:
- ✅ **Menos colunas que One-Hot**: 15 categorias → 4 colunas (log₂(15) ≈ 4)
- ✅ **Melhor para alta cardinalidade** que One-Hot Encoding
- ⚠️ **Menos interpretável** que as outras técnicas
- 📊 **Ideal quando** o número de categorias é grande mas não extremo

## 6. Comparação Visual das Técnicas

In [ ]:
# @title
# Comparando o número de colunas criadas por diferentes codificações
# para a variável 'cidade' de alta cardinalidade

# One-Hot Encoding
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

onehot_encoder = OneHotEncoder(sparse_output=False, drop='first')
onehot_encoded = onehot_encoder.fit_transform(df[[col_high_card]])
n_onehot_cols = onehot_encoded.shape[1]

# Label Encoding
label_encoder = LabelEncoder()
n_label_cols = 1  # Sempre gera apenas uma coluna

# Binary Encoding (já calculado anteriormente)
n_binary_cols = binary_encoded.shape[1]

# Comparação
comparison_df = pd.DataFrame({
    'Método': ['Label Encoding', 'Binary Encoding', 'One-Hot Encoding'],
    'Número de Colunas': [n_label_cols, n_binary_cols, n_onehot_cols]
})

print(f"Comparação do número de colunas para a variável '{col_high_card}' (cardinalidade = {df[col_high_card].nunique()}):")
print(comparison_df)
print(f"\nRedução com Binary Encoding vs One-Hot: {(1 - n_binary_cols/n_onehot_cols)*100:.1f}%")

In [ ]:
# @title
# Visualização comparativa do crescimento de dimensionalidade
import matplotlib.pyplot as plt
import numpy as np

# Comparação visual das três técnicas
cardinalities = [5, 10, 20, 50, 100]
label_cols = [1] * len(cardinalities)  # Sempre 1 coluna
onehot_cols = [n-1 for n in cardinalities]  # n-1 colunas (com drop='first')
binary_cols = [int(np.ceil(np.log2(n))) for n in cardinalities]  # log2(n) colunas

# Plotando
plt.figure(figsize=(10, 6))
x = np.arange(len(cardinalities))
width = 0.25

plt.bar(x - width, label_cols, width, label='Label Encoding', color='#2E86AB')
plt.bar(x, binary_cols, width, label='Binary Encoding', color='#A23B72')
plt.bar(x + width, onehot_cols, width, label='One-Hot Encoding', color='#F18F01')

plt.xlabel('Número de Categorias Únicas')
plt.ylabel('Número de Colunas Geradas')
plt.title('Comparação de Dimensionalidade: Label vs Binary vs One-Hot')
plt.xticks(x, cardinalities)
plt.legend()
plt.grid(axis='y', alpha=0.3)

# Adicionando valores nas barras
for i, v in enumerate(label_cols):
    plt.text(i - width, v + 0.5, str(v), ha='center')
for i, v in enumerate(binary_cols):
    plt.text(i, v + 0.5, str(v), ha='center')
for i, v in enumerate(onehot_cols):
    plt.text(i + width, v + 0.5, str(v), ha='center')

plt.tight_layout()
plt.show()

## 7. Resumo e Recomendações

### 📊 Comparação das Técnicas Fundamentais

| Técnica | Dimensões | Vantagens | Desvantagens | Quando Usar |
|---------|-----------|-----------|--------------|-------------|
| **Label Encoding** | 1 coluna | Simples, compacto | Cria ordem artificial | Árvores de decisão, variáveis ordinais |
| **One-Hot Encoding** | n-1 colunas | Remove ordem artificial | Alta dimensionalidade | Variáveis nominais, poucos valores únicos |
| **Binary Encoding** | log₂(n) colunas | Equilíbrio dimensionalidade | Menos interpretável | Alta cardinalidade, memória limitada |

### 🎯 Recomendações Práticas

1. **Para variáveis com ordem natural** (ex: tamanho P/M/G):
   - Use **Ordinal Encoding** (veremos na Parte 2)

2. **Para variáveis nominais com poucas categorias** (< 10):
   - Use **One-Hot Encoding**

3. **Para variáveis com muitas categorias** (> 50):
   - Considere **Binary Encoding** ou **Target Encoding** (Parte 2)

4. **Para algoritmos baseados em árvores**:
   - **Label Encoding** pode ser suficiente

### ➡️ Continue para a Parte 2

Na **Parte 2**, exploraremos técnicas avançadas:
- **Ordinal Encoding** para respeitar ordem natural
- **Target Encoding** com técnicas de regularização
- **Hash Encoding** para altíssima cardinalidade
- **Frequency Encoding** baseado em popularidade
- Comparação sistemática de performance